In [21]:
import os

# Create Data Lake folders
os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)

# Note: You must manually upload 'sales.csv' into the 'data_lake/bronze/' folder via the Colab file browser.

In [22]:
!pip install pyspark

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, to_date, when

# Initialize SparkSession
spark = SparkSession.builder.appName("DataLakePreprocessing").getOrCreate()

# Load Bronze data
bronze_path = "data_lake/bronze/sales.csv"
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Inspect the dataset
print(f"Original Record Count: {df.count()}")
df.printSchema()
df.show(5)

Original Record Count: 1000
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+--------------

In [24]:
from pyspark.sql.functions import col, trim, upper, when, coalesce, expr, to_date

# 1. Remove exact duplicates
df_clean = df.dropDuplicates()

# 2. Handle missing values
df_clean = df_clean.dropna(subset=["order_id", "product"])
df_clean = df_clean.fillna({"discount_percent": 0.0, "customer_name": "Unknown"})

# 3. Trim extra spaces and standardize text capitalization
df_clean = df_clean.withColumn("customer_name", upper(trim(col("customer_name"))))

# 4. Basic validation on numerical fields (Nullify clearly invalid negative values)
for num_col in ["quantity", "unit_price", "discount_percent"]:
    df_clean = df_clean.withColumn(
        num_col,
        when(col(num_col) < 0, None).otherwise(col(num_col))
    )

# 5. Convert valid dates and safely handle invalid dates
# try_to_timestamp returns NULL for non-existent dates (like Feb 31) without crashing,
# and to_date strips the timestamp to keep only the 'yyyy-MM-dd' date format.
df_clean = df_clean.withColumn(
    "order_date",
    to_date(
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'dd/MM/yyyy')"),
            expr("try_to_timestamp(order_date, 'MM/dd/yyyy')")
        )
    )
)

# Save resulting DataFrame to the Silver layer as CSV
silver_path = "data_lake/silver/preprocessed_sales/sales_silver.csv"
df_clean.write.mode("overwrite").csv(silver_path, header=True)

In [25]:
# Show row counts
print(f"Original Row Count: {df.count()}")
print(f"Processed Row Count: {df_clean.count()}")

# Read and verify Silver data
df_silver = spark.read.csv(silver_path, header=True, inferSchema=True)
df_silver.show(5)

# Briefly state operations
print("Preprocessing applied: Dropped exact duplicate rows, removed rows with missing IDs, filled remaining nulls with defaults, trimmed and uppercased customer names, nullified negative numerical values, and standardized order dates.")

Original Row Count: 1000
Processed Row Count: 990
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      SAKSHI JOSHI|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|SIDDHARTH KULKARNI|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     DIYA TRIPATHI|Competitive Exam ...|    Books|       6